In [327]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
import ast


In [328]:
ads_base = pd.read_csv('../data/ads.csv')
users_base = pd.read_csv('../data/users.csv')

users= users_base.copy()
ads = ads_base.copy()

In [329]:
users.head(1)

,user_id,age,gender,lat,lon,preferences
0,0,45,F,-33.106155,-53.58914,"[('Game_Hardware', 0.8485463411219264), ('Game..."


In [330]:
ads.head(1)

,ad_id,lat_target,lon_target,location_sensitive,max_radius_km,min_age,max_age,is_any_age,gender_target,categories,bid,budget,trust_score,estimated_conversion_rate,click_probability
0,0,1.285495,-61.053986,0,2000,0,100,True,F,"{'Game_Development': 0.03083668354262722, 'For...",15.0,15000.0,0.988041,0.015604,0.031313


In [331]:
ads["categories"].head()

0    {'Game_Development': 0.03083668354262722, 'For...
1    {'MOBA': 0.08566344789189491, 'Mobile_Gaming':...
2    {'Sports_Games': 0.49102780454698103, 'Minecra...
3    {'Game_Hardware': 0.37218099077587796, 'Open_W...
4    {'MMORPG': 0.9785586009059007, 'PC_Gaming': 0....
Name: categories, dtype: object

In [332]:
type(users.preferences)

pandas.core.series.Series

In [333]:
# CSV perde a informação de tipo, então precisamos converter as strings de volta para listas e para dicionários. Para isso, podemos usar a função ast.literal_eval, que avalia uma string como uma expressão Python e retorna o objeto correspondente.

users["preferences"] = users["preferences"].apply(ast.literal_eval)
ads['categories'] = ads['categories'].apply(ast.literal_eval)

# Features Categorias

Gerar as categorias com seus pesos

OBS: isso foi feito porque para trabalhar com a abordagem de vetores, ADS e Users devem possuir mesma dimensões

Depois desenvolver uma forma mais inteligente para não fazer à mão

In [334]:
all_categories  = [
"FPS",
"Battle_Royale",
"MOBA",
"RPG",
"MMORPG",
"Strategy",
"Simulation",
"Sports_Games",
"Racing",
"Indie_Games",
"Retro_Gaming",
"Mobile_Gaming",
"PC_Gaming",
"Console_Gaming",
"Game_Development",
"Game_Streaming",
"Esports",
"Game_Hardware",
"VR_Gaming",
"Open_World",
"League_of_Legends",
"Fortnite",
"Minecraft",
"Counter_Strike",
"World_of_Warcraft"
]

In [335]:
# enumerar as categorias para criar um índice de categoria, que será usado para construir os vetores de características dos usuários e dos anúncios.
category_index = {cat: i for i, cat in enumerate(all_categories)}


In [336]:
category_index

{'FPS': 0,
 'Battle_Royale': 1,
 'MOBA': 2,
 'RPG': 3,
 'MMORPG': 4,
 'Strategy': 5,
 'Simulation': 6,
 'Sports_Games': 7,
 'Racing': 8,
 'Indie_Games': 9,
 'Retro_Gaming': 10,
 'Mobile_Gaming': 11,
 'PC_Gaming': 12,
 'Console_Gaming': 13,
 'Game_Development': 14,
 'Game_Streaming': 15,
 'Esports': 16,
 'Game_Hardware': 17,
 'VR_Gaming': 18,
 'Open_World': 19,
 'League_of_Legends': 20,
 'Fortnite': 21,
 'Minecraft': 22,
 'Counter_Strike': 23,
 'World_of_Warcraft': 24}

In [337]:
# função para construir o vetor de características do usuário a partir de suas preferências, usando o índice de categoria para mapear as categorias para posições no vetor.

def build_user_vector(preferences, category_index):

    vector = np.zeros(len(category_index))

    for category, weight in preferences:
        if category in category_index:
            vector[category_index[category]] = weight

    return vector

In [338]:
# fazendo um teste para ver se a função build_user_vector está funcionando corretamente, construindo o vetor de características do primeiro usuário do dataset.
teste = build_user_vector(users.iloc[0]['preferences'], category_index)
display(teste)
display(users.iloc[0]['preferences'])
display(category_index)

array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.15145366,
       0.        , 0.        , 0.84854634, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ])

[('Game_Hardware', 0.8485463411219264),
 ('Game_Development', 0.15145365887807363)]

{'FPS': 0,
 'Battle_Royale': 1,
 'MOBA': 2,
 'RPG': 3,
 'MMORPG': 4,
 'Strategy': 5,
 'Simulation': 6,
 'Sports_Games': 7,
 'Racing': 8,
 'Indie_Games': 9,
 'Retro_Gaming': 10,
 'Mobile_Gaming': 11,
 'PC_Gaming': 12,
 'Console_Gaming': 13,
 'Game_Development': 14,
 'Game_Streaming': 15,
 'Esports': 16,
 'Game_Hardware': 17,
 'VR_Gaming': 18,
 'Open_World': 19,
 'League_of_Legends': 20,
 'Fortnite': 21,
 'Minecraft': 22,
 'Counter_Strike': 23,
 'World_of_Warcraft': 24}

In [339]:
len(teste)

25

In [340]:
# função análoga para o ads
# feito isso pq o ads foi salvo como dict e user como list

def build_ad_vector(categories, category_index):

    vector = np.zeros(len(category_index))

    for category, weight in categories.items():
        if category in category_index:
            vector[category_index[category]] = weight

    return vector

In [341]:
# teste para ads
teste_ad = build_ad_vector(ads.iloc[0]['categories'], category_index)
display(teste_ad)
display(ads.iloc[0]['categories'])


array([0.        , 0.        , 0.        , 0.        , 0.35456129,
       0.        , 0.        , 0.01817338, 0.        , 0.03925901,
       0.        , 0.        , 0.        , 0.        , 0.03083668,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.17101861, 0.11956046, 0.        , 0.26659057, 0.        ])

{'Game_Development': 0.03083668354262722,
 'Fortnite': 0.11956046184830019,
 'Sports_Games': 0.018173378487163622,
 'MMORPG': 0.35456128766617023,
 'League_of_Legends': 0.17101861324341172,
 'Counter_Strike': 0.266590565443359,
 'Indie_Games': 0.03925900976896807}

In [342]:
# aplcar as funções para construir os vetores de características para todos os usuários e anúncios, e armazenar esses vetores em novas colunas nos dataframes.

users['feature_vector'] = users['preferences'].apply(lambda prefs: build_user_vector(prefs, category_index))
ads['feature_vector'] = ads['categories'].apply(lambda cats: build_ad_vector(cats, category_index))

In [343]:
users.head(1)

,user_id,age,gender,lat,lon,preferences,feature_vector
0,0,45,F,-33.106155,-53.58914,"[(Game_Hardware, 0.8485463411219264), (Game_De...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


# Gerar os vetores finais 

- vai fazer a similaridade tudo junto ou vai separar??

In [344]:
# definir ordem fixa das features

CATEGORY_COLUMNS = all_categories

In [345]:
# transformar features_vector em colunas separadas
category_features_users = pd.DataFrame(
    users["feature_vector"].tolist(),
    columns=CATEGORY_COLUMNS
)

category_features_ads = pd.DataFrame(
    ads["feature_vector"].tolist(),
    columns=CATEGORY_COLUMNS
)

users = pd.concat([users, category_features_users], axis=1)
ads = pd.concat([ads, category_features_ads], axis=1)

# Gerar os vetores finais sem cidade/estado

In [346]:
category_features_users

,FPS,Battle_Royale,MOBA,RPG,MMORPG,Strategy,Simulation,Sports_Games,Racing,Indie_Games,...,Game_Streaming,Esports,Game_Hardware,VR_Gaming,Open_World,League_of_Legends,Fortnite,Minecraft,Counter_Strike,World_of_Warcraft
0,0.000000,0.000000,0.0,0.000000,0.0,0.000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.848546,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.0,0.000000,0.0,0.000,0.000000,0.717441,0.000000,0.282559,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.0,0.000000,0.0,0.000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.0,0.000000,0.0,0.000,0.040488,0.000000,0.000000,0.138106,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.149990,0.179896
4,0.000000,0.000000,0.0,0.319278,0.0,0.000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.680722,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0.000000,0.000000,0.0,0.199963,0.0,0.000,0.000000,0.000000,0.575985,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.007961
9996,0.000000,0.000000,0.0,0.000000,0.0,0.000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.537227,0.000000,0.000000
9997,0.156999,0.407886,0.0,0.000000,0.0,0.000,0.373462,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.061653,0.000000
9998,0.000000,0.574812,0.0,0.000000,0.0,0.000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.425188


In [347]:
users.iloc[0].preferences

[('Game_Hardware', 0.8485463411219264),
 ('Game_Development', 0.15145365887807363)]

In [348]:
users[category_features_users.columns].iloc[0]

FPS                  0.000000
Battle_Royale        0.000000
MOBA                 0.000000
RPG                  0.000000
MMORPG               0.000000
Strategy             0.000000
Simulation           0.000000
Sports_Games         0.000000
Racing               0.000000
Indie_Games          0.000000
Retro_Gaming         0.000000
Mobile_Gaming        0.000000
PC_Gaming            0.000000
Console_Gaming       0.000000
Game_Development     0.151454
Game_Streaming       0.000000
Esports              0.000000
Game_Hardware        0.848546
VR_Gaming            0.000000
Open_World           0.000000
League_of_Legends    0.000000
Fortnite             0.000000
Minecraft            0.000000
Counter_Strike       0.000000
World_of_Warcraft    0.000000
Name: 0, dtype: float64

In [349]:
FINAL_FEATURE_COLUMNS = (
    CATEGORY_COLUMNS
)

In [350]:
user_matrix = users[FINAL_FEATURE_COLUMNS].values
ad_matrix = ads[FINAL_FEATURE_COLUMNS].values

# PRECISA NORMALIZAR???

In [351]:
user_matrix

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.15699946, 0.40788586, 0.        , ..., 0.        , 0.06165252,
        0.        ],
       [0.        , 0.574812  , 0.        , ..., 0.        , 0.        ,
        0.425188  ],
       [0.00525721, 0.        , 0.        , ..., 0.        , 0.18760419,
        0.21895896]])

In [352]:
similarity_matrix = cosine_similarity(user_matrix, ad_matrix)

In [353]:
similarity_matrix
# cada linha representa um usuário e cada coluna representa um anúncio, e os valores são as similaridades entre eles.

array([[0.01098819, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.0634668 , 0.        , 0.6460098 , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.05764941, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.17108319, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

## Validar os resultados

In [354]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [367]:
user_index = 98

user_lat = users.iloc[user_index]["lat"]
user_lon = users.iloc[user_index]["lon"]

ads["distance_km"] = haversine(
    user_lat,
    user_lon,
    ads["lat_target"],
    ads["lon_target"]
)

tau = 200

ads["distance_score"] = np.exp(-ads["distance_km"] / tau)

ads.loc[
    ads["location_sensitive"] == 0,
    "distance_score"
] = 1

In [368]:
user_gender = users.iloc[user_index]["gender"]
user_age = users.iloc[user_index]["age"]

scores = similarity_matrix[user_index]

distance_scores = ads["distance_score"].values

final_scores = scores * distance_scores

ads_filtered = ads[
    (ads["gender_target"].isin([user_gender, "Any"])) &
    (
        (ads["min_age"].isna()) |
        ((ads["min_age"] <= user_age) & (user_age <= ads["max_age"]))
    ) &
    (
        ads["distance_km"] <= ads["max_radius_km"]
    )
]
filtered_scores = final_scores[ads_filtered.index]

top_ads = ads_filtered.index[filtered_scores.argsort()[::-1][:100]]

In [369]:
len(filtered_scores)

103

In [370]:
top_ads

Index([467, 590, 423, 945, 239,  35, 787,   0, 341, 611, 571, 172, 980,   9,
       533,  11, 993, 256, 546, 627, 696, 595, 226, 573, 317, 465,  54, 657,
       682,  80, 740, 788, 916, 792,  52,  32, 289, 294, 301, 979, 319, 130,
       280,   1, 343, 355, 376, 377, 383, 287, 949, 262, 144, 923, 928, 229,
       227, 407, 213,  63, 185, 173,  79, 925, 391, 428, 410, 597, 636, 651,
       661, 675, 684, 694, 704, 719, 721, 735, 743, 745, 754, 760, 769, 620,
       854, 420, 583, 908, 425, 782, 435, 466, 884, 472, 474, 486, 503, 883,
       881, 540],
      dtype='int64')

In [371]:
scores[top_ads]

array([8.02753962e-01, 3.25638644e-01, 4.46460344e-01, 1.05693294e-01,
       8.00656738e-02, 1.99264249e-01, 4.64499054e-02, 2.59262234e-02,
       1.20207166e-01, 1.72191300e-02, 1.66767813e-01, 1.54979026e-01,
       4.46709275e-01, 3.25638644e-01, 4.56886703e-01, 5.18409741e-03,
       3.15547164e-01, 7.94168927e-02, 2.10610225e-01, 1.42899100e-03,
       2.25203907e-02, 3.61342012e-02, 2.17727571e-01, 3.11070013e-01,
       2.54070367e-01, 9.42098170e-01, 4.92535872e-01, 3.31416359e-02,
       1.73767231e-01, 1.04165059e-01, 2.80086085e-01, 2.56426599e-03,
       1.23597037e-01, 1.44897624e-02, 1.17463852e-04, 1.62325981e-02,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
      

In [372]:
ads_base.iloc[top_ads]

,ad_id,lat_target,lon_target,location_sensitive,max_radius_km,min_age,max_age,is_any_age,gender_target,categories,bid,budget,trust_score,estimated_conversion_rate,click_probability
467,467,-26.724036,-61.152185,0,5000,28,37,False,F,"{'MMORPG': 0.2579509737056837, 'Racing': 0.015...",15.000000,15000.000000,0.976826,0.037940,0.010727
590,590,-5.341885,-56.469640,0,2000,0,100,True,F,{'Indie_Games': 1.0},15.191610,1519.160994,0.910156,0.049884,0.054168
423,423,-13.348699,-66.662187,1,5000,28,37,False,Any,"{'PC_Gaming': 0.25178707197038885, 'Fortnite':...",15.060951,7530.475537,0.642343,0.006877,0.014756
945,945,-5.214401,-58.797164,0,5000,0,100,True,Any,"{'Fortnite': 0.34426096173802456, 'Console_Gam...",27.395168,13697.583943,0.709218,0.018249,0.018508
239,239,-1.719848,-61.769476,0,2000,0,100,True,F,{'Strategy': 1.0},23.768531,11884.265275,0.878231,0.013125,0.050103
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
486,486,-23.104708,-59.621935,0,2000,0,100,True,Any,"{'Fortnite': 0.6425348889169785, 'MMORPG': 0.3...",15.000000,7500.000000,0.787384,0.026243,0.012095
503,503,-3.885905,-54.145997,1,5000,28,37,False,F,"{'FPS': 0.07755244003224197, 'VR_Gaming': 0.11...",37.746339,18873.169659,0.982224,0.032220,0.037652
883,883,-23.070543,-49.963036,1,2000,28,37,False,Any,{'RPG': 0.9999999999999999},19.556015,9778.007328,0.907373,0.049073,0.030203
881,881,-13.630540,-40.084640,1,5000,0,100,True,Any,"{'Retro_Gaming': 0.817019572253268, 'Esports':...",19.908895,1990.889480,0.846254,0.028936,0.046015


In [361]:
users.loc[0]

user_id                                                              0
age                                                                 45
gender                                                               F
lat                                                         -33.106155
lon                                                          -53.58914
preferences          [(Game_Hardware, 0.8485463411219264), (Game_De...
feature_vector       [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
FPS                                                                0.0
Battle_Royale                                                      0.0
MOBA                                                               0.0
RPG                                                                0.0
MMORPG                                                             0.0
Strategy                                                           0.0
Simulation                                                         0.0
Sports

# **OBJETIVO : score final**

score =
interest_similarity
× demographic_score
× bid
× trust_score
× estimated_conversion_rate

NÃO ESQUECER: incluir as